# Notebook 02: Graph Visualization
Visualize the heterogeneous timetabling graph built by `graph_builder.py`.

In [ ]:
import sys, json, pathlib
sys.path.insert(0, '../src')
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from graph_builder import build_hetero_graph

In [ ]:
with open('../data/indian_synthetic/instance_20c_small.json') as f:
    inst = json.load(f)
g = build_hetero_graph(inst)
print(g)
print('Node types:', g.node_types)
print('Edge types:', g.edge_types)

## Node Feature Dimensions

In [ ]:
import pandas as pd
rows = [{'node_type': nt, 'num_nodes': g[nt].x.shape[0], 'feature_dim': g[nt].x.shape[1]}
        for nt in g.node_types]
print(pd.DataFrame(rows).to_string(index=False))

## Edge Counts

In [ ]:
for et in g.edge_types:
    n_edges = g[et].edge_index.shape[1]
    print(f'{et}: {n_edges} edges')

## NetworkX Visualization (small subgraph)

In [ ]:
G = nx.DiGraph()
NODE_COLORS = {'course': '#4C9BE8', 'faculty': '#E84C4C', 'room': '#4CE87D', 'timeslot': '#E8C84C'}
for nt in g.node_types:
    n = g[nt].x.shape[0]
    for i in range(min(n, 5)):  # show max 5 nodes per type
        G.add_node(f'{nt}_{i}', node_type=nt)
# Add a sample of edges
for src_t, rel, dst_t in g.edge_types[:3]:
    ei = g[src_t, rel, dst_t].edge_index
    for k in range(min(ei.shape[1], 8)):
        s, d = ei[0,k].item(), ei[1,k].item()
        if f'{src_t}_{s}' in G and f'{dst_t}_{d}' in G:
            G.add_edge(f'{src_t}_{s}', f'{dst_t}_{d}', label=rel)
pos = nx.spring_layout(G, seed=42, k=2)
colors = [NODE_COLORS.get(G.nodes[n].get('node_type',''), '#888') for n in G.nodes]
plt.figure(figsize=(12, 8))
nx.draw(G, pos, node_color=colors, with_labels=True, font_size=7, node_size=600, arrows=True, arrowsize=15)
patches = [mpatches.Patch(color=c, label=t) for t, c in NODE_COLORS.items()]
plt.legend(handles=patches, loc='upper left')
plt.title('Heterogeneous Timetabling Graph (sample)')
plt.tight_layout()
plt.savefig('02_graph_viz.png', dpi=150)
plt.show()
print('Figure saved.')